**`03_show_land_and_building_values_3d`**

This notebook constructs an interactive 3D map of land and building values.

* **Height & Color**: Represent value per unit area (footprint or parcel). Low values are short/blue; high values are tall/red.
* **Volume**: Proportional to the total asset value (USD) since Volume = Area * Height.
* **Stacking**: Building footprints are stacked on top of their parcel (if multiple: the one with the largest overlap).
* **Ground elevation**: Pass `--elevation_recipe US_land-elevation-usgs-3dep` to ground the scene in USGS 3DEP terrain instead of sea level. Each parcel becomes a small terrain mesh whose top follows the ground and whose walls rise from it (`--elevation_mode mesh`, the default); buildings sit flat at the mean elevation under their footprint. The mesh coarsens with the number of parcels to stay under `--mesh_max_vertices`, and a city-scale scene falls back to `drape`, one polygon per parcel, which is much faster to build and render. The DEM is ingested on first use and sampled elevations are cached, so repeat runs are fast.
* **Terrain exaggeration**: `--terrain_exaggeration <N>` multiplies ground elevation only (not value height). Default 1; the test arguments use 3 so relief stays visible next to the value extrusion.
* **Basemap**: `--basemap` (`satellite`, `osm`, `topo`; `positron` and `dark_matter` need a CARTO API key).
* **Camera pitch**: starts at 45 and can tilt up to 85 (lonboard's default ceiling is 60).

Requires `lonboard` (installed via the `viz-fast` option of the openplaces `conda` environment).

To view the map interactively fullscreen:
1. Install `voila` into your `conda` environment.
2. From the repository root, run:
   ```bash
   voila notebooks/09_show/03_show_land_and_building_values_3d.ipynb
   voila notebooks\09_show\03_show_land_and_building_values_3d.ipynb
   ```
3. Press **F11** for browser fullscreen.

# Configure

In [ ]:
import argparse
import sys
import webbrowser
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from lonboard import Map

from openplaces.config import cfg
from openplaces.core.schema import AdminId
from openplaces.io.readers import get_admin
from openplaces.io.transform import convert_area_unit
from openplaces.path import path as build_path
from openplaces.recipe import get_recipe_by_id
from openplaces.viz import (
    get_admin_boundary_layer,
    get_basemap_layer,
    get_maplibre_basemap,
    get_terrain_basemap_layer,
    set_camera_pitch,
    show_value_terrain_layer,
)
from openplaces.viz.axes import add_log_ticks
from openplaces.viz.colors import DIVERGING_COLORMAPS, get_diverging_colormap
from openplaces.viz.elevation import get_elevation_datum

try:
    from IPython import get_ipython

    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

In [ ]:
# Voila tags each cell's element with `celltag_<tag>` but ships no CSS
# for it, and its renderer never runs TagRemovePreprocessor, so this
# rule is what actually hides `remove-cell`-tagged cells. Left untagged
# itself; a <style> element has no visible box.
if IN_NOTEBOOK:
    from IPython.display import HTML, display

    display(HTML('<style>.celltag_remove-cell{display:none !important;}</style>'))

In [ ]:
parser = argparse.ArgumentParser(
    description='3D value terrain (parcels + buildings) for one or more admin units'
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to visualize (e.g. "US-MA-SU US-MA-MI US-MA-NOK")',
    nargs='*',
    default=['US-MA-SU'],
)
parser.add_argument(
    '--parcel_recipe',
    help='Curated parcel recipe supplying the land-value surface.',
    default='US_parcel-openplaces-2026',
)
parser.add_argument(
    '--building_recipe',
    help='Curated footprint recipe supplying the building-value blocks. '
    'Named explicitly: a curate recipe is scoped to the whole country, so '
    'its scope says nothing about which units have output (see --missing).',
    default='US_footprint-openplaces-2026',
)
parser.add_argument(
    '--admin_level',
    help='Administrative level of --admin_ids, for boundaries and extent. '
    'Default None: derived from the IDs (a New England town is level 3, '
    'a municipality under a county tier is level 4).',
    type=int,
    default=None,
)
parser.add_argument(
    '--admin_recipe',
    help='Admin recipe supplying boundary geometry (e.g. '
    "'US-MA_admin-census-2025_admin3'). Default None: get_admin picks the "
    'most specific recipe covering the requested IDs.',
    default=None,
)
parser.add_argument(
    '--missing',
    help='How to handle admin units with no processed output for a recipe',
    default='warn',
    choices=['raise', 'warn', 'ignore'],
)
parser.add_argument(
    '--unit_system',
    help="Display units: 'imperial' (land $/ac, buildings $/sqft) or "
    "'metric' (land $/ha, buildings $/m2). Affects color scales only; "
    'height is always $/m2.',
    default='metric',
    choices=['imperial', 'metric'],
)
parser.add_argument(
    '--land_cmap',
    help='Colormap name (custom or matplotlib standard) for parcels',
    default='turbo',
)
parser.add_argument(
    '--building_cmap',
    help='Colormap name (custom or matplotlib standard) for buildings',
    default='turbo',
)
parser.add_argument(
    '--land_brightness',
    help='HSV brightness multiplier applied to the parcels colormap',
    type=float,
    default=1,
)
parser.add_argument(
    '--building_brightness',
    help='HSV brightness multiplier applied to the buildings colormap',
    type=float,
    default=1,
)
parser.add_argument(
    '--basemap',
    help='Ground-plane basemap style. Satellite by default: the CARTO '
    'styles (positron, dark_matter) now require an API key.',
    default='satellite',
    choices=['satellite', 'osm', 'positron', 'dark_matter', 'topo'],
)
parser.add_argument(
    '--elevation_recipe',
    help="DEM recipe to ground the scene in (e.g. 'US_land-elevation-usgs-3dep'). "
    'Parcels follow the DEM as a mesh; buildings sit flat at the mean '
    'elevation under their footprint. Default None: sea level.',
    default=None,
)
parser.add_argument(
    '--elevation_mode',
    help="How parcels meet the terrain: 'mesh' (a small terrain mesh per "
    "parcel, coarsened to fit --mesh_max_vertices), 'drape' (one polygon "
    "per parcel with draped vertices) or 'flat' (one elevation per parcel).",
    default='mesh',
    choices=['mesh', 'drape', 'flat'],
)
parser.add_argument(
    '--mesh_max_vertices',
    help='Vertex budget for the whole mesh scene; the grid coarsens to fit '
    'and a scene too large for a 2x2 grid per parcel falls back to drape. '
    '0 disables the budget.',
    type=int,
    default=2_000_000,
)
parser.add_argument(
    '--draped_basemap',
    help='Drape the basemap over the DEM as a colored quad mesh instead of '
    'a flat tile layer. Requires --elevation_recipe; tens of seconds per '
    'town, meant for one admin unit at a time.',
    action='store_true',
)
parser.add_argument(
    '--basemap_resolution',
    help='Grid cell size in meters for --draped_basemap. 20 keeps a street '
    'network readable; 30 is softer; 60 is unusable.',
    type=float,
    default=20.0,
)
parser.add_argument(
    '--terrain_exaggeration',
    help='Multiplier on ground elevation only, not on value height. 1 = true to scale.',
    type=float,
    default=1.0,
)
# Hand-tuned color bounds in $/ac (land) and $/sqft (buildings),
# converted to the --unit_system below.
parser.add_argument(
    '--land_vmin',
    help='Color-scale lower bound for parcels, in $/ac',
    type=float,
    default=250,
)
parser.add_argument(
    '--land_vmax',
    help='Color-scale upper bound for parcels, in $/ac',
    type=float,
    default=10_000_000,
)
parser.add_argument(
    '--building_vmin',
    help='Color-scale lower bound for buildings, in $/sqft',
    type=float,
    default=25,
)
parser.add_argument(
    '--building_vmax',
    help='Color-scale upper bound for buildings, in $/sqft',
    type=float,
    default=50_000,
)

# Test arguments

In [ ]:
args_test = (
    '--admin_ids US-PA-HU '
    '--admin_ids US-MA-CAM US-MA-SOM US-MA-BOS '
    '--land_cmap turbo '
    '--building_cmap jet '
    '--building_vmin 10 '
    '--building_vmax 2500 '
    '--basemap dark_matter '
    # "--unit_system imperial "
    # "--missing raise "
    # "--land_brightness 1.2 "
    # "--building_brightness 0.8 "
    # "--land_vmin 1000 "
    # "--land_vmax 10000000 "
    # Terrain testing: real 3DEP ground elevation (ingested automatically
    # if not already on disk), exaggerated 3x so relief on known hills is
    # actually visible -- turn back to 1 (or drop both flags) for a
    # true-to-scale render.
    '--elevation_recipe US_land-elevation-usgs-3dep '
    '--terrain_exaggeration 3 '
)

if IN_NOTEBOOK:
    args_list = [x for x in args_test.split(' ') if x]
else:
    args_list = sys.argv[1:] or [x for x in args_test.split(' ') if x]

args = parser.parse_args(args_list)
args

In [ ]:
def show_colormap_swatches(names=None, width=6, swatch_height=0.35):
    """Preview colormaps as horizontal gradient swatches."""
    if names is None:
        names = list(DIVERGING_COLORMAPS) + [
            'turbo',
            'jet',
            'nipy_spectral',
            'terrain',
            'rainbow',
            'pride',
            'CET_R1',
            'CET_R4',
        ]
    else:
        names = list(names)
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    fig, axes = plt.subplots(len(names), 1, figsize=(width, swatch_height * len(names)))
    for ax, name in zip(axes, names, strict=True):
        ax.imshow(gradient, aspect='auto', cmap=get_diverging_colormap(name))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha='right', va='center', fontsize=9)
    fig.tight_layout()
    return fig


_ = show_colormap_swatches()

# Configure display

In [ ]:
# Display area units for color scales and colorbars; height stays $/m2.
unit_system = args.unit_system

_AREA_UNITS = {
    'imperial': {'land': 'ac', 'building': 'sqft'},
    'metric': {'land': 'ha', 'building': 'm2'},
}
land_area_unit = _AREA_UNITS[unit_system]['land']
building_area_unit = _AREA_UNITS[unit_system]['building']

# Custom palette names or matplotlib colormaps.
building_cmap = get_diverging_colormap(args.building_cmap)
land_cmap = get_diverging_colormap(args.land_cmap)

# Color bounds are given in $/ac and $/sqft; convert to display units.
land_vmin = convert_area_unit(args.land_vmin, 'ac', land_area_unit)
land_vmax = convert_area_unit(args.land_vmax, 'ac', land_area_unit)
building_vmin = convert_area_unit(args.building_vmin, 'sqft', building_area_unit)
building_vmax = convert_area_unit(args.building_vmax, 'sqft', building_area_unit)

# The admin level comes from the IDs: a New England town and a county
# elsewhere are both level 3, a municipality under a county tier is 4.
# get_admin likewise picks the most specific recipe for the IDs;
# --admin_level and --admin_recipe override both.
admin_level = args.admin_level or max(
    AdminId(admin_id).get_level() for admin_id in args.admin_ids
)
admin_recipe = args.admin_recipe

# One elevation datum for the whole scene. Extruding from sea level
# lifts everything as far above the flat basemap as the land is above
# the ocean, and a tilted camera reads that height as a horizontal
# shift (h * tan(pitch)), so streets and buildings drift apart.
# Referencing to the ground under it (a low quantile of the DEM, not
# the minimum: a few bad pixels lifted Boston 77 m) leaves the relief.
# Every layer must get the same datum, or they slide against each other.
if args.elevation_recipe:
    elevation_datum = get_elevation_datum(
        get_admin(args.admin_ids, level=admin_level, recipe=admin_recipe, geom=True),
        args.elevation_recipe,
    )
    print(f'elevation datum: {elevation_datum:,.1f} m')
else:
    elevation_datum = 0.0

In [ ]:
# Map.to_html() embeds the whole dataset inline; above this many
# features the file is too large for a browser, so skip the export and
# point to Voila instead.
_TO_HTML_FEATURE_LIMIT = 50_000


def show_or_save(m, filename):
    """Render inline in a notebook; outside one, save HTML and open it."""
    if IN_NOTEBOOK:
        return m
    # The basemap tile layer has no `.table`.
    n_features = sum(
        len(layer.table)
        for layer in m.layers
        if getattr(layer, 'table', None) is not None
    )
    if n_features > _TO_HTML_FEATURE_LIMIT:
        print(
            f'{n_features:,} total features across {len(m.layers)} layer(s) -- '
            f'over the {_TO_HTML_FEATURE_LIMIT:,}-feature limit for a static '
            'HTML export (Map.to_html() embeds the entire dataset inline and '
            "won't render at this size). Skipping the export -- use Voila "
            'instead (see the intro markdown, "Viewing this map fullscreen").'
        )
        return None
    path = Path(filename).resolve()
    m.to_html(str(path), title=path.name)
    webbrowser.open(f'file://{path}')
    print(f'Saved and opened: {path}')
    return None

# Build map

## Parcels

In [ ]:
# Parcels and buildings are built in separate cells so Voila's
# "Executing N of M" counter shows which one is running.
#
# height_clip_percentile=99.9999 leaves nearly every row uncapped for
# this selection; re-check it for another one. Both layers share
# DEFAULT_ELEVATION_SCALE, which keeps $/m2 -> height consistent.
#
# elevation_mode='mesh' extrudes each parcel as a small terrain mesh:
# the top follows the ground, the outer pieces' sides are the walls.
# The mesh coarsens with the parcel count to fit mesh_max_vertices
# and a city-scale scene falls back to 'drape' automatically.
# 'drape' hands deck.gl the bare 3D ring, whose 2D triangulation tilts
# long slivers across the parcel; 'flat' gives one elevation per
# parcel. simplify_tolerance=0.5 coverage-simplifies first, so
# neighbors keep identical shared edges and their walls meet. alpha=1:
# a translucent top would show the mesh's interior sides as a grid.
parcels_3d = show_value_terrain_layer(
    args.parcel_recipe,
    args.admin_ids,
    value_column='land_value_imputed',  # real, else the street-wise imputed estimate
    area_unit=land_area_unit,
    vmin=land_vmin,
    vmax=land_vmax,
    height_clip_percentile=99.9999,
    # Drop rights-of-way (find_corridors): a road polygon is narrower
    # than a pixel at this zoom, so only its outline survives, as black
    # hairlines across the map. Parcels without a value are kept, flat
    # and gray: dropping them would punch holes in the terrain.
    drop_corridors=True,
    cmap=land_cmap,
    brightness=args.land_brightness,
    outline_width=3,
    alpha=1.0,
    elevation_recipe=args.elevation_recipe,
    elevation_mode=args.elevation_mode,
    mesh_max_vertices=args.mesh_max_vertices or None,
    simplify_tolerance=0.5,
    elevation_datum=elevation_datum,
    terrain_exaggeration=args.terrain_exaggeration,
    missing=args.missing,
)
print(
    f'{len(parcels_3d.layer.table)} parcels, '
    f'elevation_scale={parcels_3d.elevation_scale:.6g}'
)

## Admin layer

In [ ]:
# The requested units' boundaries as a translucent 3D fence. With
# elevation_recipe the fence follows the DEM and `elevation` is a wall
# height above the ground. snap_to pins its foot to the parcel walls
# beside it: parcel vertices near the boundary are inserted into the
# line with their own z, so the two do not sample the terrain at
# different points. Datum and exaggeration must match the other layers.
admin_layer = get_admin_boundary_layer(
    args.admin_ids,
    level=admin_level,
    recipe=admin_recipe,
    elevation=5,
    width=1,
    color='magenta',
    opacity=0.75,
    mode='fence',
    elevation_recipe=args.elevation_recipe,
    terrain_exaggeration=args.terrain_exaggeration,
    elevation_datum=elevation_datum,
    fill_color='magenta',
    fill_opacity=0.2,
    snap_to=parcels_3d.gdf,
)

## Buildings

In [ ]:
# stack_on=parcels_3d places each building on top of its parcel's
# value height, so this cell runs after the parcels cell.
# value_column='structure_value' is the footprint's improvement value
# (apportion_curated_values): the assessed figure, or the total minus
# the imputed land share where a land value was estimated.
# elevation_mode='flat' sits each building at the mean DEM elevation
# under its own footprint; a building's base is flat, not draped.
# Datum and exaggeration must match the parcels or buildings float.
buildings_3d = show_value_terrain_layer(
    args.building_recipe,
    args.admin_ids,
    value_column='structure_value',  # real, else (real - imputed land share)
    area_unit=building_area_unit,
    vmin=building_vmin,
    vmax=building_vmax,
    # clipped_fill_rgba=(255, 0, 255, 63),
    height_clip_percentile=99.999,
    # A footprint without a structure_value is usually a real building
    # nobody assessed separately (a shed beside the house). Float it as
    # an outline above the parcel rather than extrude it at $0.
    missing_value='ghost',
    cmap=building_cmap,
    brightness=args.building_brightness,
    outline_width=0,
    stack_on=parcels_3d,
    elevation_recipe=args.elevation_recipe,
    elevation_mode='flat',
    elevation_datum=elevation_datum,
    terrain_exaggeration=args.terrain_exaggeration,
    alpha=0.9,
    missing=args.missing,
)
print(
    f'{len(buildings_3d.layer.table)} buildings, '
    f'elevation_scale={buildings_3d.elevation_scale:.6g}'
)

# Show colorbar

In [ ]:
def show_log_colorbar(value_range, cmap, unit_suffix, subs=(1, 3)):
    """Colorbar matching a `show_value_terrain_layer` call's color ramp."""
    vmin, vmax = value_range
    norm = mpl.colors.Normalize(vmin=np.log1p(vmin), vmax=np.log1p(vmax))
    fig, ax = plt.subplots(figsize=(10, 1.4))
    cbar = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
    # `sep` sits between the number and its K/M/G; the area unit is a
    # separate `suffix`.
    add_log_ticks(
        cbar.ax,
        transform=np.log1p,
        axis='x',
        prefix='$',
        sep='',
        subs=subs,
        suffix=unit_suffix,
    )
    cbar.ax.tick_params(axis='x', rotation=90, labelsize=8)
    fig.tight_layout()
    return fig


# Use the vmin/vmax passed to show_value_terrain_layer, not each
# layer's observed value_range, or the colorbar would not match the
# map. Decades only (subs=(1,)) keep the wider parcel range legible.
_ = show_log_colorbar(
    (building_vmin, building_vmax), building_cmap, f'/{building_area_unit}', subs=(1, 3)
)
_ = show_log_colorbar(
    (land_vmin, land_vmax), land_cmap, f'/{land_area_unit}', subs=(1,)
)

# Show map

In [ ]:
# Layer order: basemap, parcels, buildings, admin fence last so it
# draws over everything. The optional outline/clipped/ghost layers are
# None unless in play. --draped_basemap lifts the ground plane to the
# DEM's own height so streets line up with the scene under a tilted
# camera; it costs a mesh build, so it stays opt-in.
if args.draped_basemap:
    basemap_layer = get_terrain_basemap_layer(
        args.admin_ids,
        level=admin_level,
        recipe=admin_recipe,
        elevation_recipe=args.elevation_recipe,
        provider=args.basemap,
        resolution=args.basemap_resolution,
        terrain_exaggeration=args.terrain_exaggeration,
        elevation_datum=elevation_datum,
    )
else:
    basemap_layer = get_basemap_layer(args.basemap)

layers = [basemap_layer, parcels_3d.layer]
# interior_layer: mesh triangles touching no parcel edge, drawn flat;
# extruding them would draw three hidden walls per triangle.
if parcels_3d.interior_layer is not None:
    layers.append(parcels_3d.interior_layer)
if parcels_3d.outline_layer is not None:
    layers.append(parcels_3d.outline_layer)
if parcels_3d.clipped_layer is not None:
    layers.append(parcels_3d.clipped_layer)
if parcels_3d.ghost_layer is not None:
    layers.append(parcels_3d.ghost_layer)
layers.append(buildings_3d.layer)
if buildings_3d.outline_layer is not None:
    layers.append(buildings_3d.outline_layer)
if buildings_3d.clipped_layer is not None:
    layers.append(buildings_3d.clipped_layer)
if buildings_3d.ghost_layer is not None:
    layers.append(buildings_3d.ghost_layer)
layers.append(admin_layer)

# Map's default MapLibre basemap is CARTO's, which asks for an API key,
# and without any MapLibre basemap lonboard drops the fullscreen and
# navigation controls. A keyless style keeps them; our tile layer
# covers it.
map_widget = Map(layers, height='100vh', basemap=get_maplibre_basemap())

# Start at 45 and allow tilting to 85 (lonboard's ceiling is 60).
# set_camera_pitch re-applies the limits after every interaction, since
# deck.gl echoes back only the camera pose and lonboard resets the
# constraints to their defaults.
set_camera_pitch(map_widget, pitch=45, max_pitch=85, min_pitch=0)

if IN_NOTEBOOK:
    # A drag handle on the widget's bottom edge, for resizing by hand.
    map_widget.layout.resize = 'vertical'
    map_widget.layout.overflow = 'hidden'
    # A pitch slider. deck.gl's ctrl/right-drag scales the pitch change by
    # how far the pointer travels toward the canvas edge, so on a tall
    # map inside a scrolling page a drag cannot always reach 0; the
    # slider sets the angle directly and follows the camera in return.
    from dataclasses import replace

    import ipywidgets as widgets
    from IPython.display import display

    pitch_slider = widgets.FloatSlider(
        value=45, min=0, max=85, step=1, description='pitch', readout_format='.0f'
    )

    def _slide_pitch(change):
        map_widget.view_state = replace(
            map_widget.view_state, pitch=change['new'], max_pitch=85, min_pitch=0
        )

    def _follow_camera(change):
        new = change['new']
        if new is not None and abs(pitch_slider.value - new.pitch) > 0.5:
            pitch_slider.value = new.pitch

    pitch_slider.observe(_slide_pitch, names='value')
    map_widget.observe(_follow_camera, names='view_state')
    display(pitch_slider)

# Written under share/ beside the recipe's other deliverables (the
# QGIS map uses the same path with .qgz), never into the repository.
# Several units roll up to their common parent directory.
_scene_admin_id = AdminId(args.admin_ids[0])
for _other in args.admin_ids[1:]:
    _other = AdminId(_other)
    _n = 0
    while (
        _n < min(_scene_admin_id.get_level(), _other.get_level())
        and _scene_admin_id.levels[_n] == _other.levels[_n]
    ):
        _n += 1
    _scene_admin_id = AdminId(*_scene_admin_id.levels[:_n])
scene_path = build_path(
    _scene_admin_id,
    get_recipe_by_id(args.parcel_recipe)['entity'],
    filename='terrain_3d',
    root=cfg.share_dir,
    default_extension='html',
)
scene_path.parent.mkdir(parents=True, exist_ok=True)
show_or_save(map_widget, scene_path)

# Inspect data

In [ ]:
from openplaces.api import get_entities

parcels = get_entities(args.parcel_recipe, args.admin_ids, geom=True)
parcels.sample(5).T

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
# from openplaces.flow import convert_to_script

# COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

# convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)